# 01 Cleaning

This notebook cleans the CredData extract for GitGuard. It reads `data/raw/credata_raw_extract.csv` and will write `data/processed/credata_clean.csv` in the format described in `docs/CLEANED_DATA.md`.

This first part loads the data and checks whether the labels line up with the extracted line text. That check comes first because it decides which rows can be trusted before any other cleaning step.

The secret values in CredData are look alikes, but GitHub push protection still treats some of them as real. For that reason the outputs in this notebook show counts and offsets only, never `LineText` or `Value`.

## Setup

We import pandas and set the path to the raw extract. The notebook runs from the `notebooks` folder, so the path goes up one level.

In [1]:
from pathlib import Path

import pandas as pd

# The notebook runs from the notebooks folder so data sits one level up
RAW_PATH = Path("..") / "data" / "raw" / "credata_raw_extract.csv"

## Load the raw extract

Every column is read as text and empty cells stay as empty strings. Without these settings pandas turns empty cells into missing values, reads text such as `NA` in `Value` as missing, and can read ids such as `00408ef6` or `1234e5` as numbers.

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
print("rows:", len(raw))
print("columns:", raw.shape[1])
print("missing cells after reading:", int(raw.isna().sum().sum()))
print(list(raw.columns))

rows: 67896
columns: 18
missing cells after reading: 0
['Id', 'FileID', 'Domain', 'RepoName', 'FilePath', 'LineStart', 'LineEnd', 'GroundTruth', 'ValueStart', 'ValueEnd', 'CryptographyKey', 'PredefinedPattern', 'Category', 'RepoId', 'FileExt', 'LineText', 'Value', 'Found']


The extract loads with every row and no missing cells, so empty values are kept as empty strings and can be counted directly.

## Labels and lines that were not found

`GroundTruth` is T for a real credential, F for a false positive and X for a test value or placeholder. `Found` is 0 when the extraction script could not find the labeled line in the downloaded file.

In [3]:
print(raw["GroundTruth"].value_counts().to_string())
print()
print(pd.crosstab(raw["GroundTruth"], raw["Found"], margins=True).to_string())

GroundTruth
F    48061
T    15753
X     4082

Found          0      1    All
GroundTruth                   
F            410  47651  48061
T             60  15693  15753
X             87   3995   4082
All          557  67339  67896


Real credentials are a minority class, so later steps need class weights or a similar fix. The rows with `Found` equal to 0 have no line text and will be dropped, as recorded in `docs/DECISIONS.md`.

## Alignment check

For a single line row with both offsets, `ValueEnd` should not be past the end of `LineText`. A labeled line that is blank is also a sign that the label points at the wrong line. We flag both cases on rows that were found.

In [4]:
found = raw[raw["Found"] == "1"].copy()
line_len = found["LineText"].str.len()
value_start = pd.to_numeric(found["ValueStart"], errors="coerce")
value_end = pd.to_numeric(found["ValueEnd"], errors="coerce")
single_line = found["LineStart"] == found["LineEnd"]
has_offsets = value_start.notna() & value_end.notna()

# A value that ends after the line ends cannot belong to that line
found["end_past_line"] = single_line & has_offsets & (value_end > line_len)
found["start_past_line"] = found["end_past_line"] & (value_start >= line_len)
found["blank_line"] = found["LineText"].str.strip() == ""
found["suspect"] = found["end_past_line"] | found["blank_line"]

summary = pd.DataFrame({
    "end past line": found.groupby("GroundTruth")["end_past_line"].sum(),
    "start past line": found.groupby("GroundTruth")["start_past_line"].sum(),
    "blank line": found.groupby("GroundTruth")["blank_line"].sum(),
    "suspect": found.groupby("GroundTruth")["suspect"].sum(),
})
summary.loc["total"] = summary.sum()
print(summary.to_string())
print()
print("rows with both end past line and blank line:", int((found["end_past_line"] & found["blank_line"]).sum()))

             end past line  start past line  blank line  suspect
GroundTruth                                                     
F                       57               49          67      112
T                      114               80          34      114
X                       41               32          12       41
total                  212              161         113      267

rows with both end past line and blank line: 58


Most rows whose value ends past the line also start past it, so the value is not on that line at all. That points to a line number that is off rather than an offset that is slightly wrong.

## Is it an encoding difference

If the offsets counted bytes instead of characters, a line with non ASCII text would look too short. We compare `ValueEnd` with the length of each flagged line in UTF 8 bytes.

In [5]:
flagged = found[found["end_past_line"]]
byte_len = flagged["LineText"].map(lambda text: len(text.encode("utf-8")))
print("flagged rows:", len(flagged))
print("flagged rows explained by byte length:", int((pd.to_numeric(flagged["ValueEnd"]) <= byte_len).sum()))
print("flagged rows with non ASCII text:", int((byte_len != flagged["LineText"].str.len()).sum()))

flagged rows: 212
flagged rows explained by byte length: 0
flagged rows with non ASCII text: 1


Byte counting explains none of the flagged rows, so the encoding is not the cause.

## Is it extra line break characters

Some tools also split lines on characters such as form feed or the Unicode line separator. If the labels counted those and our extraction did not, line numbers would drift. We count those characters inside the found lines.

In [6]:
break_chars = {
    "carriage return": "\r",
    "vertical tab": "\x0b",
    "form feed": "\x0c",
    "file group record separators": "[\x1c-\x1e]",
    "next line": "\x85",
    "line separator": "\u2028",
    "paragraph separator": "\u2029",
}
suspect_files = set(found.loc[found["suspect"], "FileID"])
for name, pattern in break_chars.items():
    has_char = found["LineText"].str.contains(pattern, regex=True)
    in_suspect = has_char & found["FileID"].isin(suspect_files)
    print(f"{name}: {int(has_char.sum())} rows, {int(in_suspect.sum())} in suspect files")

carriage return: 0 rows, 0 in suspect files


vertical tab: 0 rows, 0 in suspect files
form feed: 0 rows, 0 in suspect files
file group record separators: 0 rows, 0 in suspect files
next line: 2 rows, 0 in suspect files
line separator: 0 rows, 0 in suspect files
paragraph separator: 0 rows, 0 in suspect files


These characters do not appear in the lines of the suspect files, so they do not explain the drift. This only covers the labeled lines, because the full files stay in the Linux VM.

## Which files are affected

The suspect rows come from only a few `FileID` values, but from many more repositories than that. So a `FileID` is not unique across repositories. The extraction script finds a file by its `FileID` alone, so when two repositories share a `FileID`, the labels of both are read from only one of the copies. We count the shared ids and compare them with the suspect rows and with the rows where `Found` is 0.

In [7]:
repos_per_id = raw.groupby("FileID")["RepoId"].nunique()
shared_ids = set(repos_per_id[repos_per_id > 1].index)
raw_shared = raw["FileID"].isin(shared_ids)
found["shared_id"] = found["FileID"].isin(shared_ids)

# The part of the path after the repository folder shows whether shared ids are the same file
inner_path = raw.loc[raw_shared, "FilePath"].str.split("/", n=2).str[2]
same_path = raw.loc[raw_shared].assign(inner=inner_path).groupby("FileID")["inner"].nunique() == 1

print("FileID values in more than one repository:", len(shared_ids))
print("of which have the same path inside every repository:", int(same_path.sum()))
print("rows that use a shared FileID:", int(raw_shared.sum()))
print("rows with Found equal to 0 that use a shared FileID:", int((raw_shared & (raw["Found"] == "0")).sum()), "of", int((raw["Found"] == "0").sum()))
print()
print("found rows by shared FileID and suspect flag")
print(pd.crosstab(found["shared_id"], found["suspect"], margins=True).to_string())

FileID values in more than one repository: 78
of which have the same path inside every repository: 78
rows that use a shared FileID: 1602
rows with Found equal to 0 that use a shared FileID: 557 of 557

found rows by shared FileID and suspect flag
suspect    False  True    All
shared_id                    
False      66291     3  66294
True         781   264   1045
All        67072   267  67339


Almost every suspect row, and every row with `Found` equal to 0, uses a `FileID` that appears in more than one repository. The shared ids sit at the same path inside each repository, which looks like forks or copied files that changed between repositories.

So the misalignment comes from our extraction script, not from CredData. The script must find a file by its repository and `FileID` together, and the extract must be made again in the Linux VM. Until then, the rows that use a shared `FileID` cannot be trusted, even the ones that are not flagged, because a label read from the wrong copy can still land on a long enough line.

## Drop rows that use a shared FileID

We drop every row that uses a shared `FileID` instead of making the extract again in the Linux VM. This costs about 2 percent of the rows and keeps the extract that both people already have. The fixed `src/extract_lines.py` can bring these rows back before Phase 2. All rows with `Found` equal to 0 are among them, so this step also removes those.

In [8]:
before = len(raw)
clean = raw[~raw_shared].copy()
print("rows before:", before)
print("rows dropped:", before - len(clean))
print("rows left:", len(clean))
print("repositories left:", clean["RepoId"].nunique(), "of", raw["RepoId"].nunique())
print("Found values left:", clean["Found"].value_counts().to_dict())
print()
print("dropped rows by label")
print(raw.loc[raw_shared, "GroundTruth"].value_counts().to_string())

rows before:

 67896
rows dropped: 1602
rows left: 66294
repositories left: 335 of 337
Found values left: {'1': 66294}

dropped rows by label
GroundTruth
F    1137
T     283
X     182


Every remaining row was found in its own repository's file, so `Found` is now always 1. Two repositories had only shared ids and leave the data entirely.

## Drop the rows that are still misaligned

A few single line rows outside the shared ids still have a value that ends past the end of the line. Their labels cannot be checked, so we drop them as well.

In [9]:
line_len = clean["LineText"].str.len()
value_start = pd.to_numeric(clean["ValueStart"], errors="coerce")
value_end = pd.to_numeric(clean["ValueEnd"], errors="coerce")
single_line = clean["LineStart"] == clean["LineEnd"]

still_bad = single_line & value_end.notna() & (value_end > line_len)
blank = clean["LineText"].str.strip() == ""
print("rows with a value past the line:", int(still_bad.sum()))
print("rows pointing at a blank line:", int(blank.sum()))
clean = clean[~still_bad].copy()
print("rows left:", len(clean))

rows with a value past the line: 3
rows pointing at a blank line: 0
rows left: 66291


After this step no row points at a blank line and no single line value runs past its line, so every remaining label can be read against its own text.

## Missing values

The data has no missing cells in the pandas sense because empty cells are kept as empty strings. We count the empty cells per column to see what is missing and decide what each case means.

In [10]:
empty = (clean == "").sum()
empty_share = (empty / len(clean) * 100).round(2)
print(pd.DataFrame({"empty cells": empty, "percent": empty_share})[empty > 0].to_string())

                   empty cells  percent
ValueStart               32692    49.32
ValueEnd                 32893    49.62
CryptographyKey          65237    98.41
PredefinedPattern        65614    98.98
FileExt                   1056     1.59
Value                    33176    50.05


Only six columns have empty cells, and in each case the empty value has a meaning, so nothing is filled in.

- `ValueStart` and `ValueEnd` are empty when the whole line is marked rather than one value.
- `Value` is empty when there are no offsets or the line spans several lines, because the script only cuts a value from a single line.
- `CryptographyKey` and `PredefinedPattern` are optional notes that annotators added to a small number of rows.
- `FileExt` is empty when the file has no extension.

Filling these with a placeholder would hide that meaning, so they stay empty and later steps treat empty as its own case.

## Inconsistent formats

We check that number columns hold whole numbers, that text columns use one spelling for each value, and that the value offsets make sense.

In [11]:
for column in ["Id", "LineStart", "LineEnd", "ValueStart", "ValueEnd"]:
    filled = clean.loc[clean[column] != "", column]
    print(f"{column}: {int((~filled.str.fullmatch(r'[0-9]+')).sum())} values that are not whole numbers")
print()
print("Domain values:", clean["Domain"].value_counts().to_dict())
print("FileExt values with upper case letters:", int((clean["FileExt"] != clean["FileExt"].str.lower()).sum()))
print("Category values with an empty part:", int(clean["Category"].str.contains("^:|::|:$").sum()))
print("cells with spaces at either end, outside LineText and Value:",
      int(sum((clean[c] != clean[c].str.strip()).sum() for c in clean.columns if c not in ("LineText", "Value"))))
print()
value_start = pd.to_numeric(clean["ValueStart"], errors="coerce")
value_end = pd.to_numeric(clean["ValueEnd"], errors="coerce")
multi_line = clean["LineStart"] != clean["LineEnd"]
print("rows where ValueEnd is before ValueStart:", int((value_end < value_start).sum()),
      "of which multi line:", int(((value_end < value_start) & multi_line).sum()))
print("rows where ValueEnd equals ValueStart:", int((value_end == value_start).sum()))
print("rows with a ValueStart but no ValueEnd:", int((value_start.notna() & value_end.isna()).sum()))

Id: 0 values that are not whole numbers


LineStart: 0 values that are not whole numbers
LineEnd: 0 values that are not whole numbers
ValueStart: 0 values that are not whole numbers
ValueEnd: 0 values that are not whole numbers

Domain values: {'GitHub': 66253, 'Github': 38}


FileExt values with upper case letters: 0
Category values with an empty part: 0


cells with spaces at either end, outside LineText and Value: 0



rows where ValueEnd is before ValueStart: 10 of which multi line: 10
rows where ValueEnd equals ValueStart: 15
rows with a ValueStart but no ValueEnd: 201


The number columns are clean whole numbers, and file extensions and categories are consistent. The one inconsistent spelling is `Github` next to `GitHub` in `Domain`, which we fix below.

A `ValueEnd` before `ValueStart` only happens on multi line rows, where the value starts on the first line and ends on a later one, so these offsets are valid. Rows with an empty span or with only a start are kept as labeled, because the label still applies to the line.

`CryptographyKey` and `PredefinedPattern` are free text notes with mixed spellings such as `id` and `ID`. They are left as they are, because some notes such as `example` describe the label itself and should not become model features.

In [12]:
github_before = int((clean["Domain"] == "Github").sum())
clean["Domain"] = clean["Domain"].replace({"Github": "GitHub"})
print("Domain values changed:", github_before)
print("Domain values now:", clean["Domain"].value_counts().to_dict())

Domain values changed: 38
Domain values now: {'GitHub': 66291}


## Duplicates

The dataset documentation does not say how duplicates are handled, so we check several kinds. An exact duplicate is a row that repeats another in every column except `Id`. A location duplicate is a second label on the same line with the same value offsets.

In [13]:
columns_without_id = [c for c in clean.columns if c != "Id"]
line_key = ["RepoId", "FileID", "LineStart", "LineEnd"]
span_key = line_key + ["ValueStart", "ValueEnd"]

print("exact duplicate rows:", int(clean.duplicated(columns_without_id).sum()))
print("rows with the same line and the same value offsets:", int(clean.duplicated(span_key).sum()))
same_line = clean.duplicated(line_key, keep=False)
print("rows that share a line with another label:", int(same_line.sum()))
print("lines with more than one label:", int(clean[same_line].groupby(line_key).ngroups))
print("of which have labels that disagree:", int((clean[same_line].groupby(line_key)["GroundTruth"].nunique() > 1).sum()))

exact duplicate rows: 0
rows with the same line and the same value offsets: 0
rows that share a line with another label: 4945
lines with more than one label: 1875
of which have labels that disagree: 339


There are no exact duplicates and no line labeled twice for the same value. Lines with more than one label mark different values on the same line, for example a user name and a password, so they can have different labels and all of them are kept.

The same line text can also appear in several repositories, for example a config line copied between projects. These rows are not duplicates, because each one is a separate label in a separate file, but they matter for the train and test split.

In [14]:
by_text = clean.groupby("LineText").agg(rows=("Id", "size"), repos=("RepoId", "nunique"), labels=("GroundTruth", "nunique"))
print("distinct line texts:", len(by_text))
print("texts that appear in more than one repository:", int((by_text["repos"] > 1).sum()),
      "covering", int(by_text.loc[by_text["repos"] > 1, "rows"].sum()), "rows")
print("texts with more than one label:", int((by_text["labels"] > 1).sum()),
      "covering", int(by_text.loc[by_text["labels"] > 1, "rows"].sum()), "rows")

distinct line texts: 50490
texts that appear in more than one repository: 319 covering 1420 rows
texts with more than one label: 407 covering 1161 rows


Some line texts appear in more than one repository. A split by repository can still put the same text in both train and test, so `03_preprocessing` checks this overlap after splitting. Texts with more than one label are kept, because a label belongs to a value and its offsets, not to the whole line.

## Outliers

We look for outliers in the numeric measures of each row: the length of the line, the length of the value, the line number where the label starts, and how many lines the label spans. For each one we print percentiles, then count the rows above the usual box plot fence of the third quartile plus 1.5 times the interquartile range, and compare the share of real secrets above the fence with the share overall.

In [15]:
measures = pd.DataFrame({
    "line length": clean["LineText"].str.len(),
    "value length": clean["Value"].str.len().where(clean["Value"] != ""),
    "line number": clean["LineStart"].astype(int),
    "lines spanned": clean["LineEnd"].astype(int) - clean["LineStart"].astype(int) + 1,
})
is_real = clean["GroundTruth"] == "T"

print("percentiles")
print(measures.quantile([0.5, 0.9, 0.99, 0.999, 1.0]).round(1).to_string())
print()
rows = []
for name in measures.columns:
    values = measures[name].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    fence = q3 + 1.5 * (q3 - q1)
    above = values > fence
    rows.append({
        "measure": name,
        "fence": fence,
        "rows above": int(above.sum()),
        "percent above": round(above.mean() * 100, 2),
        "real secret percent above": round(is_real[above[above].index].mean() * 100, 1),
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("real secret percent overall:", round(is_real.mean() * 100, 1))

percentiles


       line length  value length  line number  lines spanned
0.500         58.0          30.0        255.0            1.0
0.900        478.0          64.0      12495.0            1.0
0.990       4067.0         218.0     185205.2            6.0
0.999      47761.0        1004.4     500181.9           28.0
1.000     711242.0        6368.0     917380.0          106.0

      measure  fence  rows above  percent above  real secret percent above
  line length  180.5        9995          15.08                       25.7
 value length   75.0        1481           4.47                       64.7
  line number 4121.0       11793          17.79                       29.6
lines spanned    1.0        1347           2.03                       94.4

real secret percent overall: 23.3


All four measures are strongly skewed. The median line is short, but a few lines run to hundreds of thousands of characters, and the fence marks a large share of rows for line length and line number. That is the shape of real code, such as minified JSON and long generated files, not a sign of broken rows.

The rows above the fence for value length and lines spanned are mostly real secrets. Long values and labels that cover several lines are typically private keys and similar credentials. Dropping or capping these rows would remove much of what the model needs to learn.

To see whether the most extreme rows look like errors, we count them by file type and check the one relationship that must hold: a value cannot be longer than its line.

In [16]:
very_long = measures["line length"] > 10000
print("rows with a line longer than 10,000 characters:", int(very_long.sum()),
      "in", clean.loc[very_long, "RepoId"].nunique(), "repositories")
print("their file types:", clean.loc[very_long, "FileExt"].value_counts().head(6).to_dict())
print("rows where the value is longer than the line:", int((measures["value length"] > measures["line length"]).sum()))

rows with a line longer than 10,000 characters: 247 in 20 repositories
their file types: {'txt': 69, 'js': 60, 'json': 46, 'cjs': 33, 'ndjson': 14, 'bundle': 13}
rows where the value is longer than the line: 0


The very long lines come from a few repositories and from text, JavaScript and JSON files, including bundled ones, where a whole file can sit on one line. No value is longer than its line. So the extreme values are real and none of them is an error.

We keep every row and do not cap any value. `03_preprocessing` applies a log transform to these length and position measures before scaling, so the long tail does not dominate.

## Concentration by repository and file type

Outliers can also be groups that are much bigger or much smaller than the rest. This matters for the split by repository, because one large repository in the test set can change the results a lot.

In [17]:
per_repo = clean.groupby("RepoId").size().sort_values(ascending=False)
print("repositories:", len(per_repo))
print("rows per repository:", per_repo.describe().round(1).to_dict())
print("share of rows in the largest repository:", round(per_repo.iloc[0] / len(clean) * 100, 2), "percent")
print("share of rows in the 5 largest repositories:", round(per_repo.head(5).sum() / len(clean) * 100, 2), "percent")
print("real secret percent in the largest repository:", round(is_real[clean["RepoId"] == per_repo.index[0]].mean() * 100, 1))
print()
per_ext = clean.groupby("FileExt").size()
print("file extensions:", len(per_ext))
print("extensions with fewer than 10 rows:", int((per_ext < 10).sum()), "covering", int(per_ext[per_ext < 10].sum()), "rows")

repositories: 335
rows per repository: {'count': 335.0, 'mean': 197.9, 'std': 688.2, 'min': 1.0, '25%': 34.0, '50%': 69.0, '75%': 160.5, 'max': 10212.0}
share of rows in the largest repository: 15.4 percent
share of rows in the 5 largest repositories: 33.32 percent
real secret percent in the largest repository: 50.6

file extensions: 219
extensions with fewer than 10 rows: 98 covering 282 rows


A few repositories hold a large share of the rows, and the largest has a much higher share of real secrets than the data overall. `03_preprocessing` therefore checks that the train and test sets have similar sizes and label shares after the split by repository.

Many file extensions are rare. They are kept as they are in the cleaned data, and grouping rare extensions is a feature choice for `src/features.py`.

## Save the cleaned data

We add the `label` column, drop `Found` because it is now always 1, and save the file in the format set out in `docs/CLEANED_DATA.md`.

In [18]:
clean["label"] = (clean["GroundTruth"] == "T").astype(int)
clean = clean.drop(columns=["Found"])
first = ["label", "GroundTruth", "Id", "RepoId", "RepoName", "FileID", "Domain", "FilePath", "FileExt"]
clean = clean[first + [c for c in clean.columns if c not in first]]

OUT_PATH = Path("..") / "data" / "processed" / "credata_clean.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# Rows end with a plain newline like the raw extract so the file is the same on every system
clean.to_csv(OUT_PATH, index=False, encoding="utf-8", lineterminator="\n")

# Read the file back with the agreed settings to prove it round trips
check = pd.read_csv(OUT_PATH, dtype=str, keep_default_na=False)
print("rows saved:", len(check))
print("columns:", list(check.columns))
print("file matches the data in memory:", check.equals(clean.astype(str).reset_index(drop=True)))
print()
print("label counts")
print(check["label"].value_counts().to_string())
print("share of label 1:", round((check["label"] == "1").mean() * 100, 2), "percent")

rows saved: 66291
columns: ['label', 'GroundTruth', 'Id', 'RepoId', 'RepoName', 'FileID', 'Domain', 'FilePath', 'FileExt', 'LineStart', 'LineEnd', 'ValueStart', 'ValueEnd', 'CryptographyKey', 'PredefinedPattern', 'Category', 'LineText', 'Value']
file matches the data in memory: True

label counts
label
0    50821
1    15470
share of label 1: 23.34 percent


The cleaned file reads back exactly as it was written. Real secrets make up about 23 percent of the rows, so the classes are imbalanced and later steps need class weights or a similar fix.

## Cleaning summary

This table shows how many rows each step removed, for the README and for checking the numbers in the docs.

In [19]:
after_shared = int((~raw_shared).sum())
summary = pd.DataFrame([
    {"step": "raw extract", "rows removed": 0, "rows left": before},
    {"step": "drop rows that use a shared FileID", "rows removed": before - after_shared, "rows left": after_shared},
    {"step": "drop rows whose value ends past the line", "rows removed": after_shared - len(clean), "rows left": len(clean)},
    {"step": "fix Domain spelling", "rows removed": 0, "rows left": len(clean)},
    {"step": "duplicates", "rows removed": 0, "rows left": len(clean)},
    {"step": "outliers", "rows removed": 0, "rows left": len(clean)},
])
print(summary.to_string(index=False))
print()
print("rows removed in total:", before - len(clean), "or", round((before - len(clean)) / before * 100, 2), "percent")

                                    step  rows removed  rows left
                             raw extract             0      67896
      drop rows that use a shared FileID          1602      66294
drop rows whose value ends past the line             3      66291
                     fix Domain spelling             0      66291
                              duplicates             0      66291
                                outliers             0      66291

rows removed in total: 1605 or 2.36 percent


Cleaning removes a little over 2 percent of the rows, almost all because of the shared `FileID` problem in the extraction. Missing values, duplicates and outliers are explained and kept, because in this data they carry meaning rather than noise.